# Modélisation — Adult Income (UCI)

Comparaison d'un large panel de classifieurs (`ModelRegistry`) en **validation
croisée sur le train** (`ModelEvaluator`), validation des choix de prétraitement
(**ablations**), puis **comparaison de tous les modèles**. On n'écarte aucun modèle ici :
l'**hyper-optimisation** et le **choix du modèle final** se font dans `hyperopt.ipynb`
(après tuning).

Pipeline : `DataLoader` → `PreprocessingPipeline` → `ModelRegistry` → `ModelEvaluator`.

> ⚠️ Le **test (`adult.test`) reste de côté** : l'évaluation finale se fait dans un
> notebook séparé, **après** l'hyper-optimisation. Ici, tout se joue en CV sur le train.

In [ ]:
# Recharge automatique des modules src/ modifiés sans restart du kernel
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Permet d'importer le package src/ depuis le dossier notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import warnings
# LightGBM émet un warning bénin sur les noms de colonnes via un pipeline sklearn
warnings.filterwarnings("ignore", message=".*does not have valid feature names.*")

import matplotlib.pyplot as plt

from adult_income import (
    DataLoader,
    ModelEvaluator,
    ModelRegistry,
    PreprocessingPipeline,
)

print("Imports OK")

## 1. Données & preprocessing

In [ ]:
# --- Données : train officiel (adult.data). Le test reste de côté. ---
loader = DataLoader()
X_train, y_train = loader.load_train()

# --- Preprocessing (propositions de l'AED) ---
pp = PreprocessingPipeline(scaling=True)

# Encoder la cible seulement (<=50K -> 0, >50K -> 1)
y_train_enc = pp.fit_transform_target(y_train)

print(f"Train : {X_train.shape[0]:,} obs")

## 2. Catalogue de modèles (`ModelRegistry`)

In [ ]:
# Tous les modèles disponibles dans le catalogue
print("Modèles disponibles :")
for name in ModelRegistry.available():
    print(f"  - {name}")

In [ ]:
model_names = ModelRegistry.available()
models = ModelRegistry.get_subset(model_names)
print(f"{len(models)} modèles à comparer :", list(models))

## 3. Validation des choix de prétraitement (ablations)

On vérifie que les **propositions de l'AED** tiennent face à leurs alternatives,
en **CV sur le train** (Average Precision), **un facteur à la fois** vs le baseline. On ne
touche pas au test.

**Deux sondes**, car l'effet d'une transfo dépend du modèle :
- **linéaire** (régression logistique) — *sensible* aux transfos monotones (`log1p`),
  à la binarisation et à l'encodage ;
- **à arbres** (XGBoost) — *invariante* aux transfos monotones / à l'échelle, et
  capable de trouver elle-même les seuils (zéro/non-zéro).

→ Lire les écarts **par colonne** : une transfo peut aider le linéaire et rester
inerte pour les arbres.

| Flag | Baseline (AED) | Variante |
|---|---|---|
| `drop_fnlwgt` | `True` (droppé) | `False` (gardé) |
| `unknown_category` | `True` (`Unknown`) | `False` (mode) |
| `capital_treatment` | `"hurdle"` | `"naive"` |

Deux sondes sont utilisées : une **régression logistique** (linéaire, très sensible aux transformations monotones et à la binarisation) et un **XGBoost** (arbres). En théorie les arbres sont invariants aux transfos monotones — mais on verra que la **binarisation de `capital-loss` leur fait perdre de l'information** (la magnitude), d'où l'intérêt de tester les deux familles.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
import pandas as pd

cv = StratifiedKFold(5, shuffle=True, random_state=42)

# Une sonde linéaire (sensible) + une à arbres (invariante aux transfos monotones)
probes = {
    "logistique (linéaire)": ModelRegistry.get("logistic_regression"),
    "xgboost (arbres)":      ModelRegistry.get("xgboost"),
}

def ap_par_sonde(build_pp):
    """Average Precision moyenne (CV) de chaque sonde pour une config de prétraitement."""
    return {
        nom: round(cross_val_score(build_pp().build_pipeline(probe), X_train, y_train_enc,
                                   cv=cv, scoring="average_precision", n_jobs=-1).mean(), 4)
        for nom, probe in probes.items()
    }

variants = {
    "baseline (AED)":             lambda: PreprocessingPipeline(), # version avec hurdle de base
    "+ fnlwgt":                   lambda: PreprocessingPipeline(drop_fnlwgt=False),
    "mode au lieu d'Unknown":     lambda: PreprocessingPipeline(unknown_category=False),
    "capital naïf (sans hurdle)": lambda: PreprocessingPipeline(capital_treatment="naive"),
}
pd.DataFrame({lab: ap_par_sonde(f) for lab, f in variants.items()}).T

## 4. Prétraitement affiné (suite aux ablations)

Les ablations montrent que le **hurdle sur `capital-*` dégrade les modèles à
arbres**, pour un gain nul sur le linéaire. Le meilleur modèle sera vraisemblablement un arbre → on
retient le traitement **`naive`** des `capital-*`. Les autres propositions de l'AED
(drop `fnlwgt`, catégorie `Unknown`, OneHot) sont validées et conservées.

Ce `pp_final` sert pour la **comparaison complète** et l'**hyper-optimisation** (notebook séparé).

In [ ]:
pp_final = PreprocessingPipeline(capital_treatment="naive")

# Pipelines complets : prétraitement affiné + chaque modèle
pipelines = {name: pp_final.build_pipeline(est) for name, est in models.items()}
print("Prétraitement affiné : capital=naive, occupation=onehot, fnlwgt droppé, catégorie Unknown")
print(f"{len(pipelines)} pipelines prêts pour la comparaison CV.")

## 5. Comparaison des modèles (validation croisée)

Tous les modèles sont évalués en CV stratifiée sur le **train**, avec le
**prétraitement affiné** `pp_final`. Métrique principale : **Average Precision**
(adaptée au déséquilibre).

In [ ]:
evaluator = ModelEvaluator(cv=5, random_state=42)
evaluator.cross_validate_all(pipelines, X_train, y_train_enc)

evaluator.cv_comparison_table()

In [ ]:
evaluator.plot_cv_comparison(metric="average_precision")
plt.show()

## 5 bis. Ablation — pondération des classes (vs réglage du seuil)

La cible est déséquilibrée (~24 % de `>50K`). On gère déjà ce déséquilibre par **deux
leviers** : la métrique de sélection (**Average Precision**, adaptée au déséquilibre) et le
**réglage du seuil** de décision (fait dans `hyperopt.ipynb`). La **pondération des classes**
est un **3ᵉ levier** : on teste ici s'il apporte quelque chose **en plus**.

**Pourquoi une étude et pas seulement un flag dans Optuna ?** La pondération ne réordonne
quasiment pas les exemples : elle déplace la masse de probabilité. Donc sur une métrique de
**classement** (AP, ROC-AUC) son effet est ~nul, et un tuner optimisant l'AP la rejette
silencieusement. Son effet réel se voit sur les métriques **à seuil fixe** (F1, rappel).

**Leviers par modèle** (ils ne partagent pas le même paramètre) :
- `logistic_regression`, `random_forest` → `class_weight ∈ {None, "balanced"}` ;
- `xgboost` (boosting) → `scale_pos_weight ∈ {1, ratio nég/pos ≈ 3,1}` ;
- `gradient_boosting` n'a pas de levier scalaire (seulement `sample_weight`) → écarté, xgboost
  représente déjà le boosting.

**Méthode.** Probabilités **hors-pli** (CV sur le train, sans fuite, test jamais touché).
Pour chaque config on rapporte l'AP/ROC-AUC (indépendants du seuil) et le F1/rappel **au seuil
0,50** puis **au seuil réglé** (`seuil*`, max-F1 hors-pli).

**Lecture attendue.** (1) `AP` ~plat partout → la pondération ne change pas la qualité de
classement, donc ni la sélection de modèle. (2) `pondéré @0,50` ≈ `aucun @seuil*` → pondérer
fait surtout ce que le réglage du seuil fait déjà. (3) `pondéré @seuil*` ≈ `aucun @seuil*` →
une fois le seuil réglé, la pondération est **redondante**. → On ne l'active pas dans le modèle
final ; le déséquilibre est traité par l'AP + le seuil.

In [ ]:
# Levier "balanced" pour scale_pos_weight (xgboost) = ratio négatifs / positifs
ratio = float((y_train_enc == 0).sum() / (y_train_enc == 1).sum())

weight_probes = [
    {"name": "logistic_regression",
     "pipeline": pp_final.build_pipeline(ModelRegistry.get("logistic_regression")),
     "param": "classifier__class_weight",
     "values": {"aucun": None, "balanced": "balanced"}},
    {"name": "random_forest",
     "pipeline": pp_final.build_pipeline(ModelRegistry.get("random_forest")),
     "param": "classifier__class_weight",
     "values": {"aucun": None, "balanced": "balanced"}},
    {"name": "xgboost",
     "pipeline": pp_final.build_pipeline(ModelRegistry.get("xgboost")),
     "param": "classifier__scale_pos_weight",
     "values": {"aucun": 1.0, f"pondéré (~{ratio:.1f})": ratio}},
]

evaluator.class_weight_comparison(weight_probes, X_train, y_train_enc).round(4)

## 6. Étape suivante — hyper-optimisation

La comparaison ci-dessus **classe** tous les modèles à hyperparamètres par défaut, mais
**aucun n'est écarté ici** : présélectionner des « finalistes » sur les réglages par défaut
serait risqué, car les modèles ne profitent pas tous autant du tuning (le boosting et les
forêts gagnent beaucoup, le linéaire presque rien — le classement peut donc se réorganiser).

→ **Tous les modèles optimisables passent à l'hyper-optimisation** (`hyperopt.ipynb`), et le
**modèle final est choisi après tuning**. On y vérifie d'ailleurs explicitement que le
classement reste stable entre params **par défaut** et **optimisés**.